In [ ]:
!pip install pdfplumber

In [ ]:
import os
import re
import pdfplumber #to extract the text from the pdf file
import pandas as pd
import numpy as np

In [ ]:
# Define a function to extract information from the pdf
def extract_information(pdf_path):
  with pdfplumber.open(pdf_path) as pdf:
    resume_text = ""
    for page in pdf.pages:
      resume_text = " ".join([resume_text, page.extract_text()])
  resume_text = resume_text.strip()
  return resume_text

In [ ]:
# Define a function to extract Skills, and Education
def extract_details(resume_text):
    # Define regular expressions to extract Skills & Education
    skills_pattern = r'Skills\n([\s\S]*?)(?=\n[A-Z]|$)'
    education_pattern = r'Education\n([\s\S]*?)(?=\n[A-Z][a-z]*\n|$)'

    # Get Skills & Education
    skills_match = re.findall(skills_pattern, resume_text, re.DOTALL)
    education_match = re.findall(education_pattern, resume_text, re.DOTALL)

    # Skills & Education
    if len(skills_match)!=0:
        skills = skills_match[0]
    else:
        skills_pattern = r'skills\n((?:.*)*)'
        skills_match = re.findall(skills_pattern, resume_text, re.DOTALL)
        if len(skills_match)!=0:
            skills = skills_match[0]
        else:
            skills = None

    if len(education_match)!=0:
        education = education_match[0]
    else:
        education = None

    return {
        'Skills': skills,
        'Education': education
    }

In [ ]:
%%time

data_folder = '/content/drive/MyDrive/data'
resume_data = []

# Iterate through sub-folders and PDF files
for category_folder in os.listdir(data_folder):
    category_path = os.path.join(data_folder, category_folder)
    if os.path.isdir(category_path):
        for pdf_file in os.listdir(category_path):
            if pdf_file.endswith('.pdf'):
                pdf_path = os.path.join(category_path, pdf_file)
                # print(pdf_path)
                text = extract_information(pdf_path)
                details = extract_details(text)

                # Adding Category & ID
                details['ID'] = pdf_file.replace('.pdf', '')
                details['Category'] = category_folder

                # print(f'File: [{pdf_path}]')
                # print(details, end='\n\n')
                resume_data.append(details)

print('PDF Extraction Done!')

In [ ]:
resume_df = pd.DataFrame(resume_data)
resume_df.to_csv('./pdf_extracted_skills_education.csv', index=False)


In [ ]:
resume_df.shape

In [ ]:
# Null values
resume_df.isna().sum()

In [ ]:
print(resume_df[(resume_df.Skills.isna() & resume_df.Education.isna())])

In [ ]:
# We are left with 2469 resumes after removing those 15 resumes with null data in both of them
print(resume_df[~(resume_df['Skills'].isna() & resume_df['Education'].isna())].shape)
cv_df = resume_df[~(resume_df['Skills'].isna() & resume_df['Education'].isna())].reset_index(drop=True)
cv_df.head()

In [ ]:
# New number of null values in Skills & Education Section
cv_df.isna().sum()

In [ ]:
# Null values in Skills Section
cv_df[cv_df.Skills.isna()]

In [ ]:
# Null values in Education Section
cv_df[cv_df.Education.isna()]

In [ ]:
cv_df.Category.value_counts()

In [ ]:
# We can see here the distribution of different CV categories
import matplotlib.pyplot as plt

plt.figure(figsize=(8,8))

cv_df.Category.value_counts().plot(kind='barh')

for index, value in enumerate(cv_df.Category.value_counts().values):
    plt.text(value, index, str(value))

plt.show();

In [ ]:
import torch
from datasets import load_dataset
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
import os

# Define the source folder in Google Drive
data_folder_source = '/content/drive/MyDrive/data'
# Define the destination folder in the Colab local environment
data_folder_destination = './data'

# Create the destination directory if it doesn't exist
!mkdir -p {data_folder_destination}

# Copy the contents of the Google Drive folder to the local Colab memory
!cp -r {data_folder_source}/* {data_folder_destination}/

print(f"Copied data from '{data_folder_source}' to '{data_folder_destination}'")

In [ ]:
jd_data = load_dataset('jacob-hugging-face/job-descriptions', split="train")
jd_data

In [ ]:
jd_df = pd.DataFrame(jd_data)
jd_df.head()

In [ ]:
# Sample JD

# jd_df['model_response'][0]
print(jd_df['job_description'][0])

In [ ]:
df = pd.read_csv('pdf_extracted_skills_education.csv')
df.head()

In [ ]:
df.shape

In [ ]:
def text_cleaning(text:str) -> str:
    if pd.isnull(text):
        return

    # lower-case everything
    text = text.lower().strip()

    # For removing puctuations
    translator = str.maketrans('', '', string.punctuation)

    # expand all the short-form words
    text = contractions.fix(text)

    # remove any special chars
    text = re.sub(r'http\S+|www\S+|https\S+', '', text) # Remove URLs
    text = re.sub(r'\S+@\S+', '', text) # Remove emails
    text = re.sub(r'\b\d{1,3}[-./]?\d{1,3}[-./]?\d{1,4}\b', '', text) # Remove phone numbers
    text = text.translate(translator) # Remove puctuations
    text = re.sub(r'[^a-zA-Z]', ' ', text) # Remove other non-alphanumeric characters

    return text.strip()

In [ ]:
# We have 15 Resumes where Skills & Education were not extracted -- REFER EDA notebook
# So, let's remove them
cv_df = df[~(df['Skills'].isna() & df['Education'].isna())].reset_index(drop=True)

# Filling the null values in Skills & Education with Empty String before concatinating them
cv_df = cv_df.fillna(value='')

# Let's stitch together Skills & Education, similar to given in job description.
cv_df['CV'] = cv_df['Skills'] + ' ' + cv_df['Education']

# Doing text cleaning
!pip install contractions
import string
import contractions
from tqdm.autonotebook import tqdm
tqdm.pandas()

cv_df['CV'] = cv_df['CV'].progress_apply(text_cleaning)

In [ ]:
cv_df.shape

In [ ]:
# Sample job descriptions
job_descriptions = jd_df['job_description'].apply(text_cleaning)[:15].to_list() # jd_df['job_description'][:15]

# Sample resumes (replace with your extracted resume data)
resumes = cv_df['CV'].to_list()

In [ ]:

# IMPORT LIBRARIES
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer
import numpy as np


model = SentenceTransformer('paraphrase-MiniLM-L3-v2')

# TOKENIZE AND EMBED JOB DESCRIPTIONS

job_description_embeddings = []
for description in job_descriptions:
    embeddings = model.encode(description)
    job_description_embeddings.append(embeddings)  # Directly append 1D embedding


# TOKENIZE AND EMBED RESUMES

resume_embeddings = []
for resume in resumes:
    embeddings = model.encode(resume)
    resume_embeddings.append(embeddings)  # Directly append 1D embedding


 # CONVERT TO NUMPY ARRAYS
job_description_embeddings = np.array(job_description_embeddings)
resume_embeddings = np.array(resume_embeddings)

In [ ]:
job_description_embeddings[0].shape, resume_embeddings[0].shape

In [ ]:
len(job_description_embeddings), len(resume_embeddings)

In [ ]:
# Calculate cosine similarity between job descriptions and resumes
similarity_scores = cosine_similarity(job_description_embeddings, resume_embeddings)
similarity_scores

In [ ]:
# Rank candidates for each job description based on similarity scores
num_top_candidates = 5
top_candidates = []

for i, job_description in enumerate(job_descriptions):
    candidates_with_scores = list(enumerate(similarity_scores[i]))
    candidates_with_scores.sort(key=lambda x: x[1], reverse=True)
    top_candidates_for_job = candidates_with_scores[:num_top_candidates]
    top_candidates.append(top_candidates_for_job)

# Print the top candidates for each job description
for i, job_description in enumerate(job_descriptions):
    print(f"Top candidates for JD {i+1} - Postition: {jd_df['position_title'][i]}")
    for candidate_index, score in top_candidates[i]:
        print(f"  Candidate {candidate_index + 1}")
    print()